# MovieLens 32M — retrieval fusion

Consume six independently saved candidate artifacts. This notebook never fits retrievers. Choose either all full-Train Validation candidates or all historical rank-training candidates with exactly matching boundaries, catalog, mappings, K and evaluation IDs. Mixing these artifact types fails.


## 1. Environment and paths

Place this repository's readable `src/recsys/`, `configs/` and `tests/` directories in the Google Drive project directory below. No other notebook needs to have run in this runtime. The existing dataset and split artifacts must already exist for real experiments. The source path and file hashes are checked before imports; restart the runtime if a different or edited package is cached. Dependency requirements and installation instructions are in the project README; no automatic package installation runs here.


In [ ]:
from pathlib import Path
import sys
import importlib.util
RUN_MODE = "smoke_test"
if RUN_MODE == "smoke_test":
    import os
    for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[_thread_variable] = "1"  # bound synthetic CPU native-library concurrency before imports
SUPPORTED_MODES = ('smoke_test', 'load_evaluate')
if RUN_MODE not in SUPPORTED_MODES:
    raise ValueError(f"Unsupported mode {RUN_MODE!r}; choose {SUPPORTED_MODES}")
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "personal_projects/movielens_hybrid_recsys"
)
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")
SRC_DIR = PROJECT_DIR / "src"
if not (SRC_DIR / "recsys" / "__init__.py").is_file():
    raise FileNotFoundError(f"Readable source missing: {SRC_DIR / 'recsys'}. Copy or clone the repository there.")
import hashlib
_source_hash = hashlib.sha256()
for _file in sorted((SRC_DIR / "recsys").rglob("*.py")):
    _source_hash.update(str(_file.relative_to(SRC_DIR)).encode())
    _source_hash.update(_file.read_bytes())
_loaded = sys.modules.get("recsys")
if _loaded is not None:
    if (not Path(_loaded.__file__).resolve().is_relative_to(SRC_DIR.resolve()) or
        getattr(_loaded, "__notebook_source_hash__", None) != _source_hash.hexdigest()):
        raise RuntimeError("A stale/different recsys module is cached. Restart this runtime before importing.")
sys.path = [str(SRC_DIR)] + [p for p in sys.path if p != str(SRC_DIR) and "recsys_source" not in p]
import recsys
recsys.__notebook_source_hash__ = _source_hash.hexdigest()
from recsys.runtime import environment
from recsys.smoke import run_smoke
print("Source:", recsys.__file__)
print(environment("/content" if Path("/content").exists() else "/tmp"))


In [ ]:
if RUN_MODE == "smoke_test":
    print(run_smoke(PROJECT_DIR, 'fusion'))


## 2. Configuration and saved candidate inputs

Paste the exact candidate artifact directories printed by notebooks 03–07. Run this notebook separately for Validation and for historical rank-training candidates. It never selects a latest run implicitly.


In [ ]:
import json
from recsys.config import default_config
from recsys.contracts import CHANNELS
from recsys.fusion.workflow import FusionExperiment
CONFIG = default_config()
CONFIG.update(json.loads((PROJECT_DIR / "configs" / "fusion.json").read_text()))
CANDIDATE_INPUTS = {source: None for source in CHANNELS}
# Example value: PROJECT_DIR / "results/validation/popularity/<run_id>/candidates"
FUSION_OUTPUT = None


## 3. Verify candidate and frozen-data contracts

Input manifests and checksums must match the same frozen dataset, cutoff, catalog/mappings, query-label identifier and Top-K budget. All six sources are required. Full-Train raw scores are preserved but are not assumed calibrated. The context is rebuilt from the corresponding read-only Train boundary; no retriever is loaded or trained.


In [ ]:
if RUN_MODE != "smoke_test":
    if any(path is None for path in CANDIDATE_INPUTS.values()):
        raise ValueError("Set all six explicit candidate artifact paths first.")
    fusion = FusionExperiment(PROJECT_DIR, CANDIDATE_INPUTS, CONFIG)
    print(fusion.reference["partition"], fusion.reference["evaluation_id"])


## 4. Union, provenance, cutoff features and RRF

Merge sorted Parquet streams one query at a time. Deduplicate `(userId,movieId)`, retaining every per-source raw score/rank/presence/fallback flag. Missing scores stay NaN and missing ranks use K+1. Add only cutoff-frozen history/popularity/genre features. Future feedback is attached as separate label-side fields, never as features. RRF uses `sum(present/(60+rank))`.


In [ ]:
if RUN_MODE != "smoke_test":
    FUSION_OUTPUT = fusion.run()
    print("Saved fusion features:", FUSION_OUTPUT)


## 5. Diagnostics and ablations

Report candidate counts, overlap Jaccard, unique relevant contributions, union recall ceiling, coverage and popularity concentration. RRF and every remove-one-channel ablation use the same fixed query population, including no-hit queries. Pool recall can use up to 600 union candidates and is distinct from Top-100 ranking recall.


In [ ]:
import pandas as pd
from IPython.display import display
if FUSION_OUTPUT is not None:
    summary = json.loads((FUSION_OUTPUT / "metrics.json").read_text())
    display(pd.DataFrame([{**reports[0], "model": name} for name, reports in summary["metrics"].items()]))
    display(pd.Series(summary["overlap_jaccard"], name="Jaccard"))
    display(pd.Series(summary["unique_relevant_interactions"], name="unique positive contribution"))
    print("Union recall ceiling:", summary["union_recall_ceiling"])
    print("Candidate count quantiles:", summary["union_candidate_count_quantiles"])
    print("Resources:", summary["profile"])


## 6. Share artifacts with ranking

Pass the historical `FUSION_OUTPUT` and the full-Train Validation `FUSION_OUTPUT` separately to Notebook 09. Its training example builder samples unknown contrasts from historical candidates; it does not retrain the six channels. Fusion feature tables remain Parquet streams, not one all-user DataFrame.

Ratings are observational submissions. Catalog exclusions, rating bursts, exposure bias and missing-not-at-random feedback limit interpretation. Six channels need not beat a smaller subset; compare the saved ablations. No benchmark results are supplied in this notebook.
